In [1]:
# input
metal_sites_file = "./tmp/metal_sites.tsv" # from https://github.com/wangchulab/MetalNet2/blob/main/dataset/collect/cmd.sh, step 2
new_metal_sites_file = "../../../pdb/collect_mbp/data/metal_sites.tsv"
uni_pdb_mapper_file = "./tmp/entryId-pdbIds.tsv"
# output
metal_pdb_anno_file = "./tmp/pdb_metal_anno.tsv"

In [2]:
import pandas as pd
from tqdm import tqdm

df_metal = pd.concat([pd.read_table(metal_sites_file), pd.read_table(new_metal_sites_file)])
df_uni_to_pdbs = pd.read_table(uni_pdb_mapper_file, header=None, names=['uniprot', "pdbs"])

pdb_chain_to_uniprot = dict()
for _, row in df_uni_to_pdbs.iterrows():
    uniprot = row["uniprot"]
    pdb_chains = row['pdbs'].split(",")
    for c in pdb_chains:
        pdb_chain_to_uniprot[c] = uniprot

records = []
for (pdb, resi_chain), df_chain in tqdm(df_metal.groupby(by=['pdb', 'resi_chain'])):

    df_chain: pd.DataFrame

    pdb_id = str.upper(f"{pdb}_{resi_chain}")
    posis = [] # ndb_seq_can_posi
    metal_resis = []
    df = df_chain.sort_values(by=["resi_ndb_seq_can_num"]).drop_duplicates(subset=["resi_ndb_seq_can_num"])
    for _, row in df.iterrows():
        metal_resis.append(row['metal_resi'])
        posis.append(row['resi_ndb_seq_can_num'] - 1)
        
    uniprot_id = ""
    if pdb_id in pdb_chain_to_uniprot.keys():
        uniprot_id = pdb_chain_to_uniprot[pdb_id]

    records.append({
        "seq_id": uniprot_id,
        "pdb_id": pdb_id,
        "posis": ",".join([str(i) for i in posis]),
        "metal_resis": ",".join(metal_resis),
    })

df_pdb_anno = pd.DataFrame(records)
len(df_pdb_anno)
df_pdb_anno = df_pdb_anno[df_pdb_anno['seq_id'].map(lambda x: len(x) != 0)]
len(df_pdb_anno)

100%|█████████▉| 65771/65772 [00:51<00:00, 1276.58it/s]


65771

59949

In [3]:
df_pdb_anno.to_csv(metal_pdb_anno_file, index=None, sep="\t")